In [0]:
# ── Config ────────────────────────────────────────────────────────────
runs_table    = "test_automation_runs2"
results_table = "test_automation_results2"

from pyspark.sql.functions import col
import pandas as pd, json
from collections import defaultdict

NO_DATA_PATTERNS = [
    "NO RECORDS TO TEST", "NO MATCHING TEST DATA", "DOES NOT EXIST IN THE",
    "NO RECORDS FOUND", "NO DATA AVAILABLE", "NO DATA TO TEST",
    "NO DATA EXISTS FOR", "UNRESOLVED_COLUMN", "FAILED TO SETUP DATA",
]
ERROR_PATTERNS = ["IS NOT DEFINED", "TEST CRASHED:"]

def _reclassify(row):
    s = str(row["status"]).upper().strip()
    if s == "PASS":  return "PASS"
    if s in ("NO_DATA", "ERROR"): return s
    msg = str(row.get("message", "") or "").upper()
    for p in NO_DATA_PATTERNS:
        if p in msg: return "NO_DATA"
    for p in ERROR_PATTERNS:
        if p in msg: return "ERROR"
    return s

runs_df    = spark.table(runs_table)
results_df = spark.table(results_table)

trans_runs = (runs_df
              .filter(col("run_by_automation_name") == "Transformation_Tests")
              .orderBy("run_start_datetime")
              .toPandas())

if trans_runs.empty:
    displayHTML("<p>No runs found in table.</p>")
else:
    run_ids = trans_runs["run_id"].tolist()
    results = (results_df
               .filter(col("run_id").isin(run_ids))
               .select("run_id", "status", "message", "test_field", "test_from_state", "test_name")
               .toPandas())
    results["status_upper"] = results.apply(_reclassify, axis=1)
    run_meta = (trans_runs
                .set_index("run_id")[["run_start_datetime", "state_under_test"]]
                .to_dict("index"))

    # ── Per-run totals (trend charts) ─────────────────────────────────
    run_agg = []
    for rid, grp in results.groupby("run_id"):
        meta  = run_meta.get(rid, {})
        date  = str(meta.get("run_start_datetime", ""))[:10]
        state = str(meta.get("state_under_test", ""))
        p  = int((grp["status_upper"] == "PASS").sum())
        f  = int((grp["status_upper"] == "FAIL").sum())
        e  = int((grp["status_upper"] == "ERROR").sum())
        nd = int((grp["status_upper"] == "NO_DATA").sum())
        run_agg.append({
            "run_id": rid[:8], "date": date, "state": state,
            "pass": p, "fail": f, "error": e, "nodata": nd,
            "total": p + f + e + nd,
        })
    run_agg.sort(key=lambda x: x["date"])

    # ── Per-(from_state, field) per-run: status only (for change detection) ─
    priority = {"PASS": 4, "FAIL": 3, "ERROR": 2, "NO_DATA": 1}
    field_runs = []
    for (from_state, field, rid), grp in results.groupby(["test_from_state", "test_field", "run_id"]):
        meta  = run_meta.get(rid, {})
        date  = str(meta.get("run_start_datetime", ""))[:10]
        state = str(meta.get("state_under_test", ""))
        if not date or not field or not from_state: continue
        best = max(grp["status_upper"].tolist(), key=lambda s: priority.get(s, 0))
        field_runs.append({
            "from_state": str(from_state), "field": str(field),
            "date": date, "run_state": state, "status": best,
        })
    field_runs.sort(key=lambda x: x["date"])

    # ── Fail/error messages per (from_state, field): most recent 5 entries ─
    # Stored separately to keep field_runs small. Limit messages to 90 days.
    cutoff_90d = (pd.Timestamp.now() - pd.Timedelta(days=90)).strftime("%Y-%m-%d")
    fail_detail_map = defaultdict(list)
    for (from_state, field, rid), grp in results.groupby(["test_from_state", "test_field", "run_id"]):
        bad = grp[grp["status_upper"].isin(["FAIL", "ERROR"])].head(3)
        if bad.empty: continue
        meta = run_meta.get(rid, {})
        date = str(meta.get("run_start_datetime", ""))[:10]
        if not date or date < cutoff_90d: continue
        msgs = []
        for _, fr in bad.iterrows():
            tn = str(fr.get("test_name", "") or "").strip()
            st = str(fr.get("status_upper", "") or "")
            mg = str(fr.get("message", "") or "").strip()[:120]
            if mg:
                msgs.append({"s": st, "n": tn, "m": mg})
        if msgs:
            key = str(from_state) + "||" + str(field)
            fail_detail_map[key].append({"date": date, "msgs": msgs})

    fail_details = {}
    for key, entries in fail_detail_map.items():
        entries.sort(key=lambda x: x["date"], reverse=True)
        fail_details[key] = entries[:5]

    all_states   = sorted(set(r["state"] for r in run_agg if r["state"]))

    # Escape </script> in all embedded JSON to prevent script-tag injection
    def _safe_json(obj):
        return json.dumps(obj).replace("</", "<\/")

    data_json        = _safe_json(run_agg)
    states_json      = _safe_json(all_states)
    fr_json          = _safe_json(field_runs)
    fail_details_json = _safe_json(fail_details)

    html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
  body {{font-family:Arial,sans-serif;margin:0;padding:12px;font-size:13px}}
  h2 {{color:#2c3e50;border-bottom:2px solid #2c3e50;padding-bottom:5px;margin-top:22px}}
  .boxes {{display:flex;gap:10px;flex-wrap:wrap;margin:10px 0}}
  .box {{padding:12px 18px;border-radius:6px;color:#fff;font-size:14px;text-align:center;min-width:80px}}
  .bp{{background:#27ae60}} .bf{{background:#e74c3c}} .be{{background:#7f8c8d}}
  .bn{{background:#f39c12}} .bt{{background:#2c3e50}}
  .fbar {{background:#ecf0f1;padding:8px 12px;border-radius:5px;margin:6px 0;
          display:flex;gap:8px;align-items:center;flex-wrap:wrap}}
  .fbar2 {{background:#e4edf7;padding:8px 12px;border-radius:5px;margin:4px 0;
           display:flex;gap:8px;align-items:center;flex-wrap:wrap}}
  .fbar label,.fbar2 label {{font-weight:bold;color:#2c3e50;font-size:12px}}
  .bgrp {{display:flex;gap:3px;flex-wrap:wrap}}
  .br {{padding:4px 10px;border:1px solid #bdc3c7;border-radius:3px;background:#fff;
        cursor:pointer;font-size:12px}}
  .br.active {{background:#2c3e50;color:#fff;border-color:#2c3e50}}
  .fbar select,.fbar input,.fbar2 select,.fbar2 input {{padding:4px 7px;border:1px solid #bdc3c7;border-radius:3px;font-size:12px}}
  .fbar2 input[type=date] {{min-width:115px}}
  .fbar select {{min-width:140px}}
  .btn {{padding:4px 10px;border:none;border-radius:3px;cursor:pointer;font-size:12px;color:#fff}}
  .btn-g {{background:#27ae60}} .btn-r {{background:#c0392b}} .btn-b {{background:#2c3e50}}
  .crow {{display:flex;gap:16px;flex-wrap:wrap;margin:14px 0}}
  .cwrap {{flex:1;min-width:320px;background:#fafafa;border:1px solid #ddd;border-radius:5px;padding:10px}}
  .cwrap h3 {{margin:0 0 6px;font-size:13px;color:#2c3e50}}
  canvas {{max-height:260px}}
  table {{border-collapse:collapse;font-size:12px;width:100%;margin-top:6px}}
  th {{background:#2c3e50;color:#fff;padding:4px 8px;text-align:left}}
  td {{border:1px solid #ddd;padding:3px 8px}}
  tr:nth-child(even) {{background:#f9f9f9}}
  .gd{{color:#27ae60;font-weight:bold}} .rd{{color:#e74c3c;font-weight:bold}}
  .gr{{color:#7f8c8d}} .od{{color:#f39c12;font-weight:bold}}
  .ch-fixed {{background:#d5f5e3}} .ch-regr {{background:#fadbd8}}
  .ch-new {{background:#fef9e7}} .ch-still{{background:#fafafa}}
  .badge {{display:inline-block;padding:1px 7px;border-radius:10px;font-size:11px;font-weight:bold;color:#fff}}
  .b-pass{{background:#27ae60}} .b-fail{{background:#e74c3c}}
  .b-err {{background:#7f8c8d}} .b-nd  {{background:#f39c12}}
  .arrow {{color:#aaa;font-size:15px;vertical-align:middle}}
  .msg-box {{background:#f4f4f4;padding:7px 10px;font-size:11px;font-family:monospace;
             border-left:3px solid #e74c3c;line-height:1.5;word-break:break-word}}
  .msg-fail{{color:#c0392b;font-weight:bold}} .msg-err{{color:#7f8c8d;font-weight:bold}}
  #custom-bar.on {{border:2px solid #27ae60;background:#edfaed}}
</style></head><body>
<h2>Test Trends Report</h2>
<p id="sub" style="color:#666"></p>

<div class="fbar">
  <label>Quick range:</label>
  <div class="bgrp">
    <button class="br" onclick="setR(1)"  id="r1">Last 1 day</button>
    <button class="br" onclick="setR(2)"  id="r2">Last 2 days</button>
    <button class="br" onclick="setR(7)"  id="r7">Last 7 days</button>
    <button class="br" onclick="setR(30)" id="r30">Last 30 days</button>
    <button class="br active" onclick="setR(90)" id="r90">Last 90 days</button>
    <button class="br" onclick="setR(0)"  id="r0">All time</button>
  </div>
  <label style="margin-left:8px">State:</label>
  <select id="sf" onchange="upd()"><option value="">All states</option></select>
</div>

<div class="fbar2" id="custom-bar">
  <label>Custom:</label>
  <span>This period</span>
  <input type="date" id="cp_s" title="This period — start"/>
  <span>to</span>
  <input type="date" id="cp_e" title="This period — end (blank = today)"/>
  <span style="margin-left:8px">Compare against</span>
  <input type="date" id="pp_s" title="Previous period — start"/>
  <span>to</span>
  <input type="date" id="pp_e" title="Previous period — end"/>
  <button class="btn btn-g" onclick="applyCustom()">Apply</button>
  <button class="btn btn-r" id="clr-btn" onclick="clearCustom()" style="display:none">Clear</button>
  <span id="cust-lbl" style="color:#27ae60;font-weight:bold;font-size:11px"></span>
</div>

<div class="boxes">
  <div class="box bp" id="bpass">PASS<br><b>-</b></div>
  <div class="box bf" id="bfail">FAIL<br><b>-</b></div>
  <div class="box be" id="berr">ERROR<br><b>-</b></div>
  <div class="box bn" id="bnd">NO DATA<br><b>-</b></div>
  <div class="box bt" id="btot">TOTAL<br><b>-</b></div>
</div>

<div class="crow">
  <div class="cwrap">
    <h3>Overall Results Over Time (daily totals)</h3>
    <canvas id="c1"></canvas>
  </div>
  <div class="cwrap">
    <h3>Failures by State Over Time</h3>
    <canvas id="c2"></canvas>
  </div>
</div>

<h2>Period Comparison</h2>
<p id="plabel" style="color:#666"></p>
<table style="max-width:760px">
  <thead><tr>
    <th>State</th>
    <th>This &mdash; Fail</th><th>Prev &mdash; Fail</th><th>Change</th>
    <th>This &mdash; Pass</th><th>Prev &mdash; Pass</th><th>Change</th>
  </tr></thead>
  <tbody id="ptbody"></tbody>
</table>

<h2>Status Changes: This Period vs Previous Period</h2>
<p style="color:#666">
  <b class="gd">Fixed</b> = was FAIL/ERROR, now PASS. &nbsp;
  <b class="rd">Regressed</b> = was PASS, now FAIL/ERROR. &nbsp;
  <b class="od">New</b> = no prior data, now failing. &nbsp;
  <b class="gr">Still failing</b> = bad in both. &nbsp;
  Click any row to see failure messages.
</p>
<div class="fbar" style="margin-top:4px">
  <label>Search:</label>
  <input type="text" id="chsearch" oninput="renderChanges()" placeholder="field or state..." style="min-width:160px"/>
  <label>Type:</label>
  <select id="chtype" onchange="renderChanges()">
    <option value="">All</option>
    <option value="fixed">Fixed</option>
    <option value="regressed">Regressed</option>
    <option value="new">New failure</option>
    <option value="still">Still failing</option>
  </select>
  <span id="ch-count" style="color:#7f8c8d"></span>
</div>
<table>
  <thead><tr>
    <th>From State</th><th>Field</th>
    <th>Previous</th><th></th><th>This period</th><th>Change</th>
  </tr></thead>
  <tbody id="chtbody"></tbody>
</table>

<script>
var D  = {data_json};
var FR = {fr_json};
var FD = {fail_details_json};
var STATES = {states_json};
var COLORS=['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c',
            '#e67e22','#34495e','#e91e63','#00bcd4','#8bc34a','#ff5722'];
var curR=90,ch1=null,ch2=null,_cdata=[],_cm=false;
var _cc={{cur_start:null,cur_end:null,prev_start:null,prev_end:null}};

function esc(s){{return String(s||'').replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');}}
function sm(a,k){{return a.reduce(function(s,r){{return s+(r[k]||0);}},0);}}
var PRI={{PASS:4,FAIL:3,ERROR:2,NO_DATA:1}};

function badge(s){{
  var c={{PASS:'b-pass',FAIL:'b-fail',ERROR:'b-err',NO_DATA:'b-nd'}}[s]||'b-nd';
  return '<span class="badge '+c+'">'+esc(s||'?')+'</span>';
}}

function delta(cur,prev,goodLess){{
  if(prev===null||prev===undefined)return '<span class="gr">&#x2014;</span>';
  var d=cur-prev;
  if(d===0)return '<span class="gr">0</span>';
  var good=(d<0&&goodLess)||(d>0&&!goodLess);
  return '<span class="'+(good?'gd':'rd')+'">'+(d>0?'&#x25B2;':'&#x25BC;')+' '+Math.abs(d)+'</span>';
}}

function buildSel(){{
  var sel=document.getElementById('sf');
  STATES.forEach(function(s){{
    var o=document.createElement('option');o.value=s;o.textContent=s;sel.appendChild(o);
  }});
}}

function getCuts(){{
  if(_cm)return _cc;
  if(curR<=0)return {{cur_start:null,cur_end:null,prev_start:null,prev_end:null}};
  var n=Date.now();
  return {{
    cur_start:new Date(n-curR*86400000).toISOString().slice(0,10),
    cur_end:null,
    prev_start:new Date(n-curR*2*86400000).toISOString().slice(0,10),
    prev_end:new Date(n-curR*86400000).toISOString().slice(0,10)
  }};
}}

function filtD(cuts){{
  var st=document.getElementById('sf').value;
  var d=D.filter(function(r){{return !st||r.state===st;}});
  if(cuts.cur_start)d=d.filter(function(r){{return r.date>=cuts.cur_start;}});
  if(cuts.cur_end)d=d.filter(function(r){{return r.date<=cuts.cur_end;}});
  return d;
}}
function filtPrev(cuts){{
  if(!cuts.prev_start)return [];
  var st=document.getElementById('sf').value;
  return D.filter(function(r){{
    return(!st||r.state===st)&&r.date>=cuts.prev_start&&(!cuts.prev_end||r.date<=cuts.prev_end);
  }});
}}
function filtFR(cuts){{
  var st=document.getElementById('sf').value;
  var d=FR.filter(function(r){{return !st||r.run_state===st;}});
  if(cuts.cur_start)d=d.filter(function(r){{return r.date>=cuts.cur_start;}});
  if(cuts.cur_end)d=d.filter(function(r){{return r.date<=cuts.cur_end;}});
  return d;
}}
function filtFRPrev(cuts){{
  if(!cuts.prev_start)return [];
  var st=document.getElementById('sf').value;
  return FR.filter(function(r){{
    return(!st||r.run_state===st)&&r.date>=cuts.prev_start&&(!cuts.prev_end||r.date<=cuts.prev_end);
  }});
}}

function aggDate(data){{
  var bd={{}};
  data.forEach(function(r){{
    if(!bd[r.date])bd[r.date]={{pass:0,fail:0,error:0,nodata:0}};
    bd[r.date].pass+=r.pass;bd[r.date].fail+=r.fail;
    bd[r.date].error+=r.error;bd[r.date].nodata+=r.nodata;
  }});
  var dates=Object.keys(bd).sort();
  return {{dates:dates,bd:bd}};
}}
function aggState(data){{
  var sts=[],bds={{}};
  data.forEach(function(r){{
    if(sts.indexOf(r.state)<0)sts.push(r.state);
    if(!bds[r.date])bds[r.date]={{}};
    bds[r.date][r.state]=(bds[r.date][r.state]||0)+r.fail;
  }});
  sts.sort();
  var dates=Object.keys(bds).sort();
  return {{dates:dates,states:sts,bds:bds}};
}}

function computeChanges(cuts){{
  var curFR=filtFR(cuts),prevFR=filtFRPrev(cuts);
  var curB={{}},prevB={{}};
  curFR.forEach(function(r){{
    var k=r.from_state+'||'+r.field;
    if(!curB[k]||(PRI[r.status]||0)>(PRI[curB[k]]||0))curB[k]=r.status;
  }});
  prevFR.forEach(function(r){{
    var k=r.from_state+'||'+r.field;
    if(!prevB[k]||(PRI[r.status]||0)>(PRI[prevB[k]]||0))prevB[k]=r.status;
  }});
  var keys=Object.keys(curB);
  Object.keys(prevB).forEach(function(k){{if(keys.indexOf(k)<0)keys.push(k);}});
  var changes=[];
  keys.forEach(function(k){{
    var pts=k.split('||'),fs=pts[0],fld=pts[1];
    var cur=curB[k]||null,prev=prevB[k]||null;
    var bad=function(s){{return s==='FAIL'||s==='ERROR';}};
    var type;
    if(cur==='PASS'&&bad(prev))      type='fixed';
    else if(bad(cur)&&prev==='PASS') type='regressed';
    else if(bad(cur)&&!prev)         type='new';
    else if(bad(cur)&&bad(prev))     type='still';
    else return;
    /* Collect messages from FD within the current period */
    var fdMsgs=[];
    var entries=FD[k]||[];
    entries.forEach(function(e){{
      if(cuts.cur_start&&e.date<cuts.cur_start)return;
      if(cuts.cur_end&&e.date>cuts.cur_end)return;
      if(e.msgs)e.msgs.forEach(function(m){{fdMsgs.push(m);}});
    }});
    if(!fdMsgs.length){{
      /* fall back to any stored message, ignoring date filter */
      entries.forEach(function(e){{if(e.msgs)e.msgs.forEach(function(m){{fdMsgs.push(m);}});}});
    }}
    fdMsgs=fdMsgs.slice(0,8);
    changes.push({{from_state:fs,field:fld,cur:cur,prev:prev,type:type,msgs:fdMsgs}});
  }});
  var ord={{regressed:0,new:1,still:2,fixed:3}};
  changes.sort(function(a,b){{
    var d=ord[a.type]-ord[b.type];
    if(d!==0)return d;
    return a.from_state<b.from_state?-1:1;
  }});
  return changes;
}}

function toggleDet(i){{
  var r=document.getElementById('chd'+i);
  if(r)r.style.display=r.style.display==='none'?'table-row':'none';
}}

function renderChanges(){{
  var srch=(document.getElementById('chsearch').value||'').toLowerCase();
  var tf=document.getElementById('chtype').value;
  var out='',shown=0,idx=0;
  var tLbl={{fixed:'Fixed &#x2713;',regressed:'Regressed &#x26A0;',new:'New failure',still:'Still failing'}};
  var tCls={{fixed:'ch-fixed',regressed:'ch-regr',new:'ch-new',still:'ch-still'}};
  _cdata.forEach(function(c){{
    if(tf&&c.type!==tf)return;
    if(srch&&c.field.toLowerCase().indexOf(srch)<0&&c.from_state.toLowerCase().indexOf(srch)<0)return;
    var detId='chd'+idx;
    var hm=c.msgs&&c.msgs.length>0;
    out+='<tr class="'+tCls[c.type]+'" onclick="toggleDet('+idx+')" style="cursor:pointer">'
      +'<td>'+esc(c.from_state)+'</td>'
      +'<td>'+esc(c.field)+(hm?' <span style="color:#aaa;font-size:10px">[+]</span>':'')+'</td>'
      +'<td>'+(c.prev?badge(c.prev):'<span class="gr">&#x2014;</span>')+'</td>'
      +'<td class="arrow">&#x2192;</td>'
      +'<td>'+(c.cur?badge(c.cur):'<span class="gr">&#x2014;</span>')+'</td>'
      +'<td><b>'+tLbl[c.type]+'</b></td>'
      +'</tr>';
    out+='<tr id="'+detId+'" style="display:none"><td colspan="6">';
    if(hm){{
      out+='<div class="msg-box">';
      c.msgs.forEach(function(m){{
        var mc=m.s==='FAIL'?'msg-fail':'msg-err';
        out+='<div><span class="'+mc+'">['+esc(m.s)+'] '+esc(m.n)+'</span>: '+esc(m.m)+'</div>';
      }});
      out+='</div>';
    }} else {{
      out+='<div style="padding:6px;color:#999;font-size:11px">No failure messages in the last 90 days for this field.</div>';
    }}
    out+='</td></tr>';
    shown++;idx++;
  }});
  document.getElementById('chtbody').innerHTML=out||'<tr><td colspan="6" style="color:#999;padding:8px">No status changes in selected period.</td></tr>';
  document.getElementById('ch-count').textContent=shown+' of '+_cdata.length+' shown';
}}

function upd(){{
  var cuts=getCuts();
  var data=filtD(cuts),prev=filtPrev(cuts);
  var tp=sm(data,'pass'),tf=sm(data,'fail'),te=sm(data,'error'),tn=sm(data,'nodata'),tt=sm(data,'total');
  document.getElementById('bpass').innerHTML='PASS<br><b>'+tp+'</b>';
  document.getElementById('bfail').innerHTML='FAIL<br><b>'+tf+'</b>';
  document.getElementById('berr').innerHTML='ERROR<br><b>'+te+'</b>';
  document.getElementById('bnd').innerHTML='NO DATA<br><b>'+tn+'</b>';
  document.getElementById('btot').innerHTML='TOTAL<br><b>'+tt+'</b>';

  var a1=aggDate(data);
  var ds1=[
    {{label:'Pass',data:a1.dates.map(function(d){{return a1.bd[d].pass;}}),borderColor:'#27ae60',backgroundColor:'rgba(39,174,96,0.07)',tension:0.3,fill:false,pointRadius:3}},
    {{label:'Fail',data:a1.dates.map(function(d){{return a1.bd[d].fail;}}),borderColor:'#e74c3c',backgroundColor:'rgba(231,76,60,0.07)',tension:0.3,fill:false,pointRadius:3}},
    {{label:'Error',data:a1.dates.map(function(d){{return a1.bd[d].error;}}),borderColor:'#7f8c8d',backgroundColor:'rgba(127,140,141,0.07)',tension:0.3,fill:false,pointRadius:3}},
    {{label:'No Data',data:a1.dates.map(function(d){{return a1.bd[d].nodata;}}),borderColor:'#f39c12',backgroundColor:'rgba(243,156,18,0.07)',tension:0.3,fill:false,pointRadius:3}}
  ];
  if(ch1)ch1.destroy();
  ch1=new Chart(document.getElementById('c1'),{{
    type:'line',data:{{labels:a1.dates,datasets:ds1}},
    options:{{responsive:true,interaction:{{mode:'index',intersect:false}},
      plugins:{{legend:{{position:'bottom',labels:{{font:{{size:11}}}}}}}},
      scales:{{x:{{ticks:{{font:{{size:10}},maxRotation:45}}}},y:{{beginAtZero:true}}}}}}
  }});

  var a2=aggState(data);
  var ds2=a2.states.map(function(s,i){{
    return {{label:s,
      data:a2.dates.map(function(d){{return(a2.bds[d]&&a2.bds[d][s])?a2.bds[d][s]:0;}}),
      borderColor:COLORS[i%COLORS.length],backgroundColor:'transparent',tension:0.3,fill:false,pointRadius:3}};
  }});
  if(ch2)ch2.destroy();
  ch2=new Chart(document.getElementById('c2'),{{
    type:'line',data:{{labels:a2.dates,datasets:ds2}},
    options:{{responsive:true,interaction:{{mode:'index',intersect:false}},
      plugins:{{legend:{{position:'bottom',labels:{{font:{{size:10}},boxWidth:10}}}}}},
      scales:{{x:{{ticks:{{font:{{size:10}},maxRotation:45}}}},y:{{beginAtZero:true,title:{{display:true,text:'Failures'}}}}}}}}
  }});

  var sf=document.getElementById('sf').value,sts=sf?[sf]:STATES;
  var rows='',ocf=0,opf=0,ocp=0,opp=0;
  var hasPrev=!!(cuts.prev_start);
  sts.forEach(function(s){{
    var cr=data.filter(function(r){{return r.state===s;}}),
        pr=prev.filter(function(r){{return r.state===s;}});
    var cf=sm(cr,'fail'),pf=sm(pr,'fail'),cp=sm(cr,'pass'),pp=sm(pr,'pass');
    if(cf+pf+cp+pp===0)return;
    ocf+=cf;opf+=pf;ocp+=cp;opp+=pp;
    rows+='<tr><td><b>'+esc(s)+'</b></td>'
      +'<td class="rd">'+cf+'</td>'
      +'<td class="rd">'+(hasPrev?pf:'&#x2014;')+'</td>'
      +'<td>'+(hasPrev?delta(cf,pf,true):'&#x2014;')+'</td>'
      +'<td class="gd">'+cp+'</td>'
      +'<td class="gd">'+(hasPrev?pp:'&#x2014;')+'</td>'
      +'<td>'+(hasPrev?delta(cp,pp,false):'&#x2014;')+'</td></tr>';
  }});
  if(rows){{
    rows='<tr style="background:#ecf0f1;font-weight:bold"><td>TOTAL</td>'
      +'<td class="rd">'+ocf+'</td><td class="rd">'+(hasPrev?opf:'&#x2014;')+'</td>'
      +'<td>'+(hasPrev?delta(ocf,opf,true):'&#x2014;')+'</td>'
      +'<td class="gd">'+ocp+'</td><td class="gd">'+(hasPrev?opp:'&#x2014;')+'</td>'
      +'<td>'+(hasPrev?delta(ocp,opp,false):'&#x2014;')+'</td></tr>'+rows;
  }}
  document.getElementById('ptbody').innerHTML=rows||'<tr><td colspan="7">No data.</td></tr>';

  var rangeDesc='';
  if(_cm){{
    rangeDesc='Custom: '+(_cc.cur_start||'?')+' to '+(_cc.cur_end||'today');
  }} else {{
    rangeDesc=curR>0?('Last '+curR+' day'+(curR===1?'':'s')):'All time';
  }}
  document.getElementById('sub').textContent=D.length+' runs total | '+rangeDesc+' | '+data.length+' runs in view';
  var pDesc='';
  if(!hasPrev){{
    pDesc='No comparison period active (use quick buttons or set custom dates).';
  }} else if(_cm){{
    pDesc='This period: '+(_cc.cur_start||'?')+' to '+(_cc.cur_end||'today')+' vs previous: '+(_cc.prev_start||'?')+' to '+(_cc.prev_end||'?')+'. Green=improving, red=worsening.';
  }} else {{
    pDesc='Comparing last '+curR+' days vs previous '+curR+' days. Green=improving, red=worsening.';
  }}
  document.getElementById('plabel').textContent=pDesc;

  _cdata=computeChanges(cuts);
  renderChanges();
}}

function setR(d){{
  curR=d;
  ['r1','r2','r7','r30','r90','r0'].forEach(function(id){{document.getElementById(id).classList.remove('active');}});
  var m={{1:'r1',2:'r2',7:'r7',30:'r30',90:'r90',0:'r0'}};
  if(m[d])document.getElementById(m[d]).classList.add('active');
  _cm=false;
  document.getElementById('clr-btn').style.display='none';
  document.getElementById('custom-bar').classList.remove('on');
  document.getElementById('cust-lbl').textContent='';
  upd();
}}

function applyCustom(){{
  var cs=document.getElementById('cp_s').value;
  if(!cs){{alert('Set at least a This-period start date.');return;}}
  _cm=true;
  _cc={{
    cur_start:cs||null,
    cur_end:document.getElementById('cp_e').value||null,
    prev_start:document.getElementById('pp_s').value||null,
    prev_end:document.getElementById('pp_e').value||null
  }};
  ['r1','r2','r7','r30','r90','r0'].forEach(function(id){{document.getElementById(id).classList.remove('active');}});
  document.getElementById('clr-btn').style.display='';
  document.getElementById('custom-bar').classList.add('on');
  document.getElementById('cust-lbl').textContent='Custom active';
  upd();
}}

function clearCustom(){{
  _cm=false;
  ['cp_s','cp_e','pp_s','pp_e'].forEach(function(id){{document.getElementById(id).value='';}});
  document.getElementById('clr-btn').style.display='none';
  document.getElementById('custom-bar').classList.remove('on');
  document.getElementById('cust-lbl').textContent='';
  setR(curR);
}}

buildSel();upd();
</script>
</body></html>"""
    displayHTML(html)
